This notebook is the downloaded version of google colab notebook which could be found here along with all the cell outputs: https://colab.research.google.com/drive/1YOIJL6dOA1_auTzcOygk0evp9rkay4Di?usp=sharing

## Imports

In [ ]:
import os
import json
from google.colab import userdata
from openai import OpenAI
import gradio as gr

## Defining Tool

In [ ]:
system_message = """
You are a dataset generation planner.

Return ONLY valid JSON.

Schema:

{
  "task": "regression | classification | clustering",
  "num_rows": int,
  "columns": [
    {
      "name": string,
      "type": "int" | "float" | "categorical" | "text" | "boolean",
      "params": {
        // int/float → {"min": number, "max": number}
        // categorical → {"values": [list of strings]}
        // text → {"pattern": string}
        // boolean → {}
      }
    }
  ],
  "target": string or null,
  "target_type": "float" | "categorical" | null,
  "relationships": [
    "expressions like: price = year*300 - mileage*200 + noise(0,1000)"
  ],
  "class_distribution": {
    "class_name": probability
  },
  "cluster_config": {
    "num_clusters": int,
    "method": "gaussian" | "separable"
  }
}

Rules:

- Always include num_rows
- Always include at least 2–5 columns

TASK RULES:

1. Regression:
   - target MUST be present
   - target_type MUST be "float"
   - MUST include at least one relationship

2. Classification:
   - target MUST be present
   - target_type MUST be "categorical"
   - MUST include class_distribution
   - class_distribution values should sum to ~1

3. Clustering:
   - target MUST be null
   - target_type MUST be null
   - MUST include cluster_config
   - cluster_config.num_clusters should be 2–6

GENERAL RULES:

- Infer realistic feature names and ranges
- Ensure relationships use only defined columns
- Do NOT leave fields empty
- Use reasonable defaults if user input is incomplete

ONLY return valid JSON. NO explanations.
"""

In [ ]:
import json
from openai import OpenAI
openai_api_key = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key = openai_api_key)
def llm_generate_spec(user_input):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_input}
        ]
    )
    print(response.choices[0].message.content)
    return json.loads(response.choices[0].message.content)

In [ ]:
def generate_column(col, n):
    import numpy as np

    t = col["type"]
    p = col.get("params", {})

    if t == "int":
        return np.random.randint(p.get("min", 0), p.get("max", 100), n)

    elif t == "float":
        return np.random.uniform(p.get("min", 0), p.get("max", 100), n)

    elif t == "categorical":
        return np.random.choice(p.get("values", ["A", "B"]), n)

    elif t == "boolean":
        return np.random.choice([True, False], n)

    elif t == "text":
        return [f"{col['name']}_{i}" for i in range(n)]

In [ ]:
def safe_eval(expr, df):
    import numpy as np

    allowed = {
        "noise": lambda mean, std: np.random.normal(mean, std, len(df)),
        "np": np
    }

    return eval(expr, {"__builtins__": {}}, {**df.to_dict("series"), **allowed})

In [ ]:
def handle_regression(df, spec):
    for rel in spec.get("relationships", []):
        left, right = rel.split("=")
        df[left.strip()] = safe_eval(right.strip(), df)

    return df

In [ ]:
def handle_classification(df, spec):
    import numpy as np

    target = spec["target"]
    dist = spec.get("class_distribution", {"A": 0.5, "B": 0.5})

    classes = list(dist.keys())
    probs = list(dist.values())

    probs = np.array(probs) / np.sum(probs)

    df[target] = np.random.choice(classes, size=len(df), p=probs)

    return df

In [ ]:
def handle_unsupervised(df, spec):
    import numpy as np

    cluster_cfg = spec.get("cluster_config", {})
    num_clusters = cluster_cfg.get("num_clusters", 3)  # fallback

    cluster_ids = np.random.randint(0, num_clusters, len(df))

    for col in df.columns:
        if df[col].dtype != "object":
            df[col] += cluster_ids * np.random.uniform(5, 20)

    return df

In [ ]:
def run_spec(spec):
    import pandas as pd

    n = spec["num_rows"]

    data = {}

    # generate base columns (excluding target)
    for col in spec["columns"]:
        if col["name"] != spec.get("target"):
            data[col["name"]] = generate_column(col, n)

    df = pd.DataFrame(data)

    task = spec.get("task", "unsupervised")

    if task == "regression":
        df = handle_regression(df, spec)

    elif task == "classification":
        df = handle_classification(df, spec)

    else:
        df = handle_unsupervised(df, spec)

    return df

In [ ]:
def dataset_pipeline(modified_request: str):
    spec = llm_generate_spec(modified_request)
    df = run_spec(spec)

    file_path = "dataset.csv"
    df.to_csv(file_path, index=False)

    return {
        "preview": df.head().to_dict(),
        "file_path": file_path
    }

## Testing the tool

In [ ]:
test_run = dataset_pipeline("""Create a clustering dataset with 800 rows for customer segmentation.
Features include income, spending score, gender,City, age.
Generate 4 distinct clusters.""")

In [ ]:
test_run

In [ ]:
import pandas as pd
df = pd.read_csv(test_run['file_path'])
df.head()

In [ ]:
df.shape

## Adding Tool functionality

In [ ]:
pipeline_function = {
    "name": "dataset_pipeline",
    "description": "Generate a synthetic dataset based on a structured natural language request. The tool creates the dataset, returns a preview of the data, and provides a file path to download the CSV.",
    "parameters": {
        "type": "object",
        "properties": {
            "modified_request": {
                "type": "string",
                "description": "A clear and complete natural language instruction describing the dataset. It should include task type (regression, classification, or clustering), number of rows, feature names, and target variable if applicable."
            }
        },
        "required": ["modified_request"]
    }
}

In [ ]:
tools = [{"type": "function", "function": pipeline_function}]

In [ ]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "dataset_pipeline":
            arguments = json.loads(tool_call.function.arguments)
            request = arguments.get('modified_request')
            data_dict = dataset_pipeline(request)
            responses.append({
            "role": "tool",
            "content": json.dumps(data_dict),
            "tool_call_id": tool_call.id
        })
    return responses

## Gradio UI

In [ ]:
BRAIN_SYSTEM_PROMPT = """
You are an assistant that helps users design synthetic datasets.

Your job:
1. Understand the dataset requirements
2. Ask clarifying questions if needed
3. Once sufficient details are available, call the dataset_pipeline tool

-----------------------
REQUIRED INFORMATION
-----------------------

Before calling the tool, ensure the following are known or reasonably inferred:

- Dataset domain (cars, healthcare, finance, etc.)
- Task type:
  - regression (numeric target)
  - classification (categorical target)
  - clustering / unsupervised (no target)
- Number of rows
- Feature names (at least 2–5 relevant features)
- Target variable (if supervised)

Optional but helpful:
- Relationships (e.g., “price depends on mileage and year”)
- Feature types (numeric/categorical)
- Class distribution (for classification)

If required information is missing, ask a clear follow-up question.

-----------------------
TOOL CALL INSTRUCTIONS
-----------------------

When ready, call the dataset_pipeline tool with a SINGLE, well-structured natural language instruction that includes all relevant details.

DO NOT:
- Generate datasets yourself
- Output JSON
- Call the tool with incomplete or vague input

-----------------------
EXAMPLES (VERY IMPORTANT)
-----------------------

BAD tool input (too vague):
"cars dataset"

BAD tool input (missing task and target):
"create dataset with mileage and year"

GOOD tool input (regression):
"Create a regression dataset with 500 rows for predicting car prices.
Features include mileage, year, fuel type, and engine size.
Target is price, which increases with year and decreases with mileage."

GOOD tool input (classification):
"Create a classification dataset with 1000 rows for predicting customer churn.
Features include age, monthly usage, contract type, and number of support calls.
Target is churn (yes/no) with roughly balanced classes."

GOOD tool input (clustering):
"Create a clustering dataset with 800 rows for customer segmentation.
Features include income, spending score, and age.
Generate 4 distinct clusters."

-----------------------
BEHAVIOR RULES
-----------------------

- Prefer making reasonable assumptions instead of asking too many questions
- Keep tool input concise but complete
- Ensure clarity and structure in the instruction
- Only call the tool when confident the dataset can be generated

Be helpful, precise, and efficient.

Assume when the tool is called, the user is automatically displayed the dataset preview and an option to download the dataset.
"""

In [ ]:
from google.colab import userdata
key = userdata.get('OPENAI_API_KEY')
openai = OpenAI(api_key=key)
MODEL = "gpt-4.1-mini"

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": BRAIN_SYSTEM_PROMPT}] + history + [{"role": "user", "content": message}]

    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools
    )

    preview = None
    file_path = None

    while response.choices[0].finish_reason == "tool_calls":
        message_obj = response.choices[0].message
        responses = handle_tool_calls(message_obj)

        # 👇 Extract tool output here
        for r in responses:
            content = json.loads(r["content"])
            preview = content.get("preview")
            file_path = content.get("file_path")

        messages.append(message_obj)
        messages.extend(responses)

        response = openai.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )

    final_text = response.choices[0].message.content

    return final_text, preview, file_path

In [ ]:
import gradio as gr
import pandas as pd

def format_preview(preview):
    if preview is None:
        return None
    return pd.DataFrame(preview)

In [ ]:
with gr.Blocks() as demo:
    chatbot = gr.Chatbot(type="messages")
    msg = gr.Textbox(label="Enter your request")

    preview_df = gr.Dataframe(label="Dataset Preview", visible=False)
    file_output = gr.File(label="Download Dataset", visible=False)

    def respond(message, history):
      text, preview, file_path = chat(message, history)

      history.append({"role": "user", "content": message})
      history.append({"role": "assistant", "content": text})

      if file_path:  # ✅ tool was called
          return (
              history,
              gr.update(value=format_preview(preview), visible=True),
              gr.update(value=file_path, visible=True)
          )
      else:  # ❌ no dataset generated
          return (
              history,
              gr.update(visible=False),
              gr.update(visible=False)
          )

    msg.submit(respond, [msg, chatbot], [chatbot, preview_df, file_output])

demo.launch()

## Using open source models for tool LLM - json output

In [ ]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6

In [ ]:
LLAMA = "meta-llama/Meta-Llama-3.1-8B-Instruct"
PHI = "microsoft/Phi-4-mini-instruct"
GEMMA = "google/gemma-3-270m-it"
QWEN = "Qwen/Qwen3-4B-Instruct-2507"
DEEPSEEK = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

In [ ]:
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import gc

In [ ]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [ ]:
# Quantization Config - this allows us to load the model into memory and use less memory

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [ ]:
messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": """Create a clustering dataset with 800 rows for customer segmentation.
Features include income, spending score, gender,City, age.
Generate 4 distinct clusters."""}
  ]

In [ ]:
MODEL_CACHE = {}

def load_model(model_name, quant=True):
    key = (model_name, quant)

    if key in MODEL_CACHE:
        return MODEL_CACHE[key]

    print(f"🔄 Loading model: {model_name} | quant={quant}")

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
    tokenizer.pad_token = tokenizer.eos_token

    if quant:
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=quant_config,
            device_map="auto"
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto"
        )

    MODEL_CACHE[key] = (tokenizer, model)
    return tokenizer, model

In [ ]:
def generate(model_name, messages, quant=True, max_new_tokens=1024):
    tokenizer, model = load_model(model_name, quant)

    input_ids = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True
    ).to(model.device)

    attention_mask = torch.ones_like(input_ids)

    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_tokens = outputs[0][input_ids.shape[-1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response

In [ ]:
print(generate(GEMMA, messages, quant=False))

In [ ]:
print(generate(PHI, messages, quant=True))

In [ ]:
print(generate(QWEN, messages, quant=True))

In [ ]:
print(generate(DEEPSEEK, messages, quant=True))

In [ ]:
print(generate(LLAMA, messages, quant=True))